In [ ]:
import os

# ----------------------- Azure OpenAI Configuration -----------------------

AZURE_FOUNDRY_API_KEY = "YOUR_AZURE_FOUNDRY_KEY"
AZURE_FOUNDRY_ENDPOINT = "https://sustainability-resource.openai.azure.com/"
AZURE_FOUNDRY_API_VERSION = "2024-02-15-preview"
AZURE_FOUNDRY_CHAT_DEPLOYMENT = "grok-3-mini" 

# Set as environment variables
os.environ["AZURE_FOUNDRY_API_KEY"] = AZURE_FOUNDRY_API_KEY
os.environ["AZURE_FOUNDRY_ENDPOINT"] = AZURE_FOUNDRY_ENDPOINT
os.environ["AZURE_FOUNDRY_API_VERSION"] = AZURE_FOUNDRY_API_VERSION
os.environ["AZURE_FOUNDRY_CHAT_DEPLOYMENT"] = AZURE_FOUNDRY_CHAT_DEPLOYMENT

print("✅ Configuration loaded successfully!")
print(f"Endpoint: {AZURE_FOUNDRY_ENDPOINT}")
print(f"Deployment: {AZURE_FOUNDRY_CHAT_DEPLOYMENT}")
print(f"API Version: {AZURE_FOUNDRY_API_VERSION}")

✅ Configuration loaded successfully!
Endpoint: https://sustainability-resource.openai.azure.com/
Deployment: grok-3-mini
API Version: 2024-02-15-preview


In [3]:
pip install -q openai pandas tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import csv
import os
import re
import time
from openai import AzureOpenAI
from IPython.display import display, Markdown

# ----------------------- Load Configuration -----------------------
AZURE_FOUNDRY_API_KEY = os.environ.get("AZURE_FOUNDRY_API_KEY")
AZURE_FOUNDRY_ENDPOINT = os.environ.get("AZURE_FOUNDRY_ENDPOINT")
AZURE_FOUNDRY_API_VERSION = os.environ.get("AZURE_FOUNDRY_API_VERSION", "2024-02-15-preview")
AZURE_FOUNDRY_CHAT_DEPLOYMENT = os.environ.get("AZURE_FOUNDRY_CHAT_DEPLOYMENT", "gpt-5-mini")

if not AZURE_FOUNDRY_API_KEY or not AZURE_FOUNDRY_ENDPOINT:
    raise ValueError(" Please run Cell 1 first to set AZURE_FOUNDRY_API_KEY and AZURE_FOUNDRY_ENDPOINT.")

# Initialize Azure OpenAI client
client = AzureOpenAI(
    api_key=AZURE_FOUNDRY_API_KEY,
    api_version=AZURE_FOUNDRY_API_VERSION,
    azure_endpoint=AZURE_FOUNDRY_ENDPOINT,
)

print("✅ Azure OpenAI client initialized successfully!")

MAX_RETRIES = 3

# Output file configuration
OUTPUT_DIRECTORY = "/path/to/your/output/folder"  

# ----------------------- IMPROVED PROMPT TEMPLATE -----------------------
SDG_PROMPT_TEMPLATE = """You are an expert SDG classifier. Your task is to classify text according to the UN's 17 Sustainable Development Goals.

**CRITICAL RULES:**
1. Base your classification ONLY on what is EXPLICITLY stated in the text
2. DO NOT make assumptions or inferences beyond what is written
3. DO NOT classify based on what "could be" or "might be" related
4. Maximum 3 SDGs - only those with DIRECT, EXPLICIT connection
5. If no strong SDG connection exists in the text → classify as NOSDG

**THE 17 UN SDGs:**
SDG1: No Poverty - Eradicating poverty in all forms
SDG2: Zero Hunger - Food security, nutrition, sustainable agriculture
SDG3: Good Health and Well-Being - Health services, disease prevention, mental health
SDG4: Quality Education - Inclusive, equitable education and lifelong learning
SDG5: Gender Equality - Women's empowerment, gender rights
SDG6: Clean Water and Sanitation - Water access, sanitation, hygiene
SDG7: Affordable and Clean Energy - Renewable energy, energy access, efficiency
SDG8: Decent Work and Economic Growth - Employment, labor rights, economic growth
SDG9: Industry, Innovation, Infrastructure - Infrastructure, industrialization, innovation
SDG10: Reduced Inequalities - Economic/social inequality reduction
SDG11: Sustainable Cities and Communities - Urban development, housing, transportation
SDG12: Responsible Consumption and Production - Sustainable production, waste reduction
SDG13: Climate Action - Climate change mitigation and adaptation
SDG14: Life Below Water - Ocean conservation, marine resources
SDG15: Life on Land - Ecosystem protection, biodiversity, forests
SDG16: Peace, Justice, Strong Institutions - Rule of law, governance, justice access
SDG17: Partnerships for the Goals - Global cooperation for SDGs

**STEP-BY-STEP CLASSIFICATION PROCESS:**

**STEP 1: EXTRACT EXPLICIT CONTENT**
List the concrete facts, actions, topics, or programs explicitly mentioned:
- What specific activities/programs/issues are described?
- What populations or sectors are explicitly mentioned?
- What outcomes or impacts are directly stated?
- What keywords or phrases directly relate to SDG themes?

**STEP 2: CHECK FOR SDG RELEVANCE**
For each SDG, ask:
- Does the text EXPLICITLY mention topics central to this SDG?
- Are there DIRECT actions/programs/outcomes related to this SDG's core goals?
- Can I point to specific words/phrases that clearly connect to this SDG?

If you cannot answer YES with specific evidence from the text → EXCLUDE that SDG

**STEP 3: APPLY STRICT INCLUSION CRITERIA**
Only include an SDG if the text shows:
✓ Direct mention of the SDG's core issues (poverty, hunger, health, education, etc.)
✓ Explicit programs/actions advancing that SDG's objectives
✓ Clear outcomes/impacts related to that SDG's targets
✓ Specific data/evidence tied to that SDG's focus areas

EXCLUDE an SDG if:
✗ Connection requires assumptions or reading between lines
✗ Relationship is indirect or tangential
✗ Generic or vague connection (e.g., "economic activity" ≠ automatically SDG8)
✗ Only mentioned in passing without substantive focus

**STEP 4: SELECT TOP SDGs (Maximum 3)**
- Rank SDGs by strength of explicit textual evidence
- Include only those with STRONG, DIRECT evidence
- When in doubt → exclude it
- Fewer SDGs is better than over-classification

**CLASSIFY AS NOSDG IF:**
- Text is purely commercial/financial with no sustainability focus
- No explicit connection to development, social, or environmental topics
- Content is technical/procedural without SDG-relevant outcomes
- Generic business/organizational content without SDG themes

**OUTPUT FORMAT:**

STEP1_EXPLICIT_CONTENT: [List key facts/topics/actions explicitly stated]

STEP2_SDG_EVIDENCE: [For each potentially relevant SDG, cite specific text evidence or state "no direct evidence"]

STEP3_FINAL_REASONING: [Explain which SDGs have strongest explicit evidence and why others are excluded]

JUSTIFICATION: [One concise sentence summarizing the classification]

SDG_CLASSIFICATION: [SDG#, SDG#, SDG# OR NOSDG]

**EXAMPLES:**

---
**Example 1:**
Text: "The solar energy program trained 200 women in rural areas to install and maintain solar panels, providing clean electricity to 50 villages."

STEP1_EXPLICIT_CONTENT: Solar energy program, training women, rural areas, solar panel installation/maintenance, clean electricity to villages

STEP2_SDG_EVIDENCE:
- SDG7: Direct mention of "solar energy," "clean electricity," renewable energy access
- SDG5: Explicit focus on training women, women's empowerment through skills
- SDG4: Training/skills development mentioned

STEP3_FINAL_REASONING: SDG7 is primary (renewable energy is the main focus). SDG5 is secondary (explicit women's empowerment component). SDG4 is weaker (training is mentioned but education/learning is not the primary focus). Limit to top 2.

JUSTIFICATION: Renewable energy program with explicit women empowerment component
SDG_CLASSIFICATION: SDG7, SDG5

---
**Example 2:**
Text: "Company Q3 revenue reached $3.2B, up 18% YoY, driven by consumer electronics and cloud services expansion."

STEP1_EXPLICIT_CONTENT: Company revenue, financial growth, consumer electronics sales, cloud services

STEP2_SDG_EVIDENCE:
- SDG8: Economic growth mentioned, but no focus on decent work, employment quality, or labor rights
- SDG9: Industry/technology mentioned, but no focus on innovation for development or infrastructure

STEP3_FINAL_REASONING: This is purely financial reporting. While economic activity is mentioned, there's no explicit focus on decent work, sustainable economic development, or innovation for social benefit. No SDG is directly addressed.

JUSTIFICATION: Financial metrics only, no explicit sustainability or development focus
SDG_CLASSIFICATION: NOSDG

---
**Example 3:**
Text: "The agricultural cooperative distributed drought-resistant seeds to 5,000 smallholder farmers, provided training on sustainable farming practices, and established fair-trade market access, increasing food production by 40% and farmer incomes by 35%."

STEP1_EXPLICIT_CONTENT: Agricultural cooperative, drought-resistant seeds, smallholder farmers, sustainable farming training, fair-trade market access, increased food production (40%), increased farmer income (35%)

STEP2_SDG_EVIDENCE:
- SDG2: Explicit food production increase, agricultural support, food security
- SDG8: Fair-trade access, income increase for farmers (decent work/economic growth)
- SDG12: Sustainable farming practices explicitly mentioned (responsible production)

STEP3_FINAL_REASONING: All three SDGs have direct textual evidence. SDG2 is primary (food production/agriculture focus). SDG8 and SDG12 are supported by explicit mentions of fair-trade/income and sustainable practices.

JUSTIFICATION: Agricultural program directly improving food security, farmer incomes, and sustainable production
SDG_CLASSIFICATION: SDG2, SDG8, SDG12

---

**TEXT TO CLASSIFY:**
{text}

**YOUR RESPONSE (Follow the format above):**
"""

# ----------------------- LLM Classification with Improved Parsing -----------------------
def normalize_sdg_format(sdg_string):
    """
    Normalize SDG format to SDG1, SDG2, etc.
    Handles: SDG01, sdg1, SDG 1, etc. -> SDG1
    """
    if not sdg_string or sdg_string.strip().upper() == "NOSDG":
        return "NOSDG"

    # Split by comma and process each SDG
    sdgs = [s.strip() for s in sdg_string.split(',')]
    normalized = []

    for sdg in sdgs:
        # Extract number from various formats
        match = re.search(r'(\d+)', sdg)
        if match:
            num = int(match.group(1))  # Convert to int to remove leading zeros
            if 1 <= num <= 17:  # Validate SDG number
                normalized.append(f"SDG{num}")
        elif sdg.strip().upper() == "NOSDG":
            normalized.append("NOSDG")

    return ", ".join(normalized) if normalized else "NOSDG"

def classify_text(text, retries=MAX_RETRIES):
    """
    Classify text using LLM with improved error handling and reasoning extraction.
    No token limit to allow reasoning models to think as deeply as needed.
    No temperature setting for maximum model compatibility.
    """
    for attempt in range(retries):
        try:
            # Build API call parameters with minimal configuration for broad model compatibility
            api_params = {
                "model": AZURE_FOUNDRY_CHAT_DEPLOYMENT,
                "messages": [
                    {"role": "system", "content": "You are an expert SDG classifier. You must think step-by-step and base classifications ONLY on explicitly stated content. Follow the exact output format provided."},
                    {"role": "user", "content": SDG_PROMPT_TEMPLATE.format(text=text)}
                ]
            }
            
            # Note: NOT setting max_completion_tokens or temperature - let the model use defaults
            response = client.chat.completions.create(**api_params)

            # Extract content
            llm_output = response.choices[0].message.content

            # Check for empty response
            if not llm_output or llm_output.strip() == "":
                reasoning = getattr(response.choices[0].message, 'reasoning_content', None)
                if reasoning:
                    print(f"⚠️ Model returned empty content despite having reasoning.")
                return "Model returned empty content", "Empty response", "NOSDG", ""

            llm_output = llm_output.strip()

            # Extract reasoning steps for transparency
            reasoning_steps = []
            for step in ["STEP1_EXPLICIT_CONTENT", "STEP2_SDG_EVIDENCE", "STEP3_FINAL_REASONING"]:
                match = re.search(rf"{step}:\s*(.+?)(?=\n(?:STEP\d|JUSTIFICATION|SDG_CLASSIFICATION)|$)", 
                                llm_output, re.IGNORECASE | re.DOTALL)
                if match:
                    reasoning_steps.append(f"{step}: {match.group(1).strip()}")

            reasoning_text = "\n\n".join(reasoning_steps) if reasoning_steps else "No detailed reasoning captured"

            # Extract JUSTIFICATION
            justification_match = re.search(
                r"JUSTIFICATION:\s*(.+?)(?:\n|$)",
                llm_output,
                re.IGNORECASE
            )
            justification = justification_match.group(1).strip() if justification_match else "No justification provided"

            # Extract SDG_CLASSIFICATION
            sdg_match = re.search(
                r"SDG_CLASSIFICATION:\s*(.+?)(?:\n|$)",
                llm_output,
                re.IGNORECASE
            )
            sdg_classification = sdg_match.group(1).strip() if sdg_match else "NOSDG"

            # Normalize SDG format
            sdg_classification = normalize_sdg_format(sdg_classification)

            return llm_output, justification, sdg_classification, reasoning_text

        except Exception as e:
            print(f"❌ Attempt {attempt + 1} failed: {e}")
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
            else:
                return "LLM Classification Failed", f"Error: {str(e)}", "NOSDG", ""

    return "LLM Classification Failed", "Max retries exceeded", "NOSDG", ""

# ----------------------- CSV Processing with Enhanced Output -----------------------
def process_csv_llm_only(input_csv_path, output_dir=None):
    """
    Process CSV and classify SDGs with detailed reasoning output.
    """
    # Determine output directory
    if output_dir is None:
        output_dir = OUTPUT_DIRECTORY

    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Generate output filename
    input_filename = os.path.basename(input_csv_path)
    base_name = os.path.splitext(input_filename)[0]
    output_filename = f"{AZURE_FOUNDRY_CHAT_DEPLOYMENT}_{base_name}_SDG_Classified.csv"
    output_csv_path = os.path.join(output_dir, output_filename)

    print(f"📂 Input file: {input_csv_path}")
    print(f"📂 Output file: {output_csv_path}")
    print(f"🤖 Model: {AZURE_FOUNDRY_CHAT_DEPLOYMENT}")
    print(f"🔧 Token limit: UNLIMITED (reasoning models can use as many tokens as needed)")
    print("="*80)

    with open(input_csv_path, "r", encoding="utf-8", newline="") as infile, \
         open(output_csv_path, "w", encoding="utf-8", newline="") as outfile:

        reader = csv.DictReader(infile)
        fieldnames = ["id", "text", "SDG_Classification", "Justification", "Reasoning_Steps", "LLM_Output"]
        writer = csv.DictWriter(outfile, fieldnames=fieldnames)
        writer.writeheader()

        total_rows = 0
        successful_classifications = 0

        for idx, row in enumerate(reader, start=1):
            text = row.get("text", "").strip()
            total_rows += 1
            print(f"\n📝 Processing row {idx}")
            print(f"   Text preview: {text[:100]}..." if len(text) > 100 else f"   Text: {text}")

            if not text:
                writer.writerow({
                    "id": row.get("id", ""),
                    "text": "",
                    "SDG_Classification": "NOSDG",
                    "Justification": "Empty text",
                    "Reasoning_Steps": "N/A",
                    "LLM_Output": "Empty text"
                })
                continue

            llm_output, justification, sdg_classification, reasoning_steps = classify_text(text)

            if sdg_classification != "NOSDG" or "Failed" not in justification:
                successful_classifications += 1

            print(f"   ✓ SDGs: {sdg_classification}")
            print(f"   ✓ Justification: {justification[:100]}...")

            writer.writerow({
                "id": row.get("id", ""),
                "text": text,
                "SDG_Classification": sdg_classification,
                "Justification": justification,
                "Reasoning_Steps": reasoning_steps,
                "LLM_Output": llm_output
            })

    print("\n" + "="*80)
    print(f"✅ Classification complete!")
    print(f"📊 Statistics:")
    print(f"   Total rows processed: {total_rows}")
    print(f"   Successful classifications: {successful_classifications}")
    print(f"   Success rate: {(successful_classifications/total_rows*100):.1f}%")
    print(f"📄 Output saved to: {output_csv_path}")
    return output_csv_path

# ----------------------- Run -----------------------
# Configure your paths here
input_csv_path = r"D:\Stevens\003 PhD\Fall 25\AAI 646\Final_Project\code\input\split_1.csv"
output_directory = r"D:\Stevens\003 PhD\Fall 25\AAI 646\Final_Project\code\results"

# Run the classification
output_csv = process_csv_llm_only(input_csv_path, output_directory)
print(f"\n✅ Final output saved to: {output_csv}")

✅ Azure OpenAI client initialized successfully!
📂 Input file: D:\Stevens\003 PhD\Fall 25\AAI 646\Final_Project\code\input\split_1.csv
📂 Output file: D:\Stevens\003 PhD\Fall 25\AAI 646\Final_Project\code\results\grok-3-mini_split_1_SDG_Classified.csv
🤖 Model: grok-3-mini
🔧 Token limit: UNLIMITED (reasoning models can use as many tokens as needed)

📝 Processing row 1
   Text preview: • showcase impact stories emerging from Griffith University staff initiatives. Report scope and boun...
   ✓ SDGs: SDG17
   ✓ Justification: The text explicitly references commitment to the UN Global Compact, a partnership mechanism directly...

📝 Processing row 2
   Text: Total by gender and degree of functional diversity 19 41 19 25 0 2 2 1 6 12 4 6 80 57
   ✓ SDGs: NOSDG
   ✓ Justification: The text is a simple data summary with no explicit links to any Sustainable Development Goals....

📝 Processing row 3
   Text preview: 13 SUSTAINABILITY HIGHLIGHTS AT THE UNIVERSITY OF HELSINKI IN 2023 GOVERNANCE AND M